# BER v8 + sibling-graph features (Kaggle, 2x T4)

Runs your friend's pipeline unchanged (`block.py`, `dense.py`, `cross.py`, `features.py`,
`decode.py`) through a new driver, **`run_graph.py`**, that adds one thing: after Stage-A
pruning, for every (query, candidate-S1) pair it looks at the *other* queries sharing that
same candidate S1 and computes embedding-cosine-based **sibling features** (cross-source
corroboration + same-source crowding) — see `graph.py`. Those become extra columns fed into
the same Stage B/C LightGBM matcher, alongside the existing ~80 features.

**What changed vs. your friend's `run.py`:** an import of `graph.py`, a new
`build_graph_features()` call right after each `build_pruned()`, `GRAPH_COLS` added to
`matcher_cols()`, and `load_chunk()` joins the new graph parquet the same way it already
joins the cross-encoder scores. Nothing else was touched.

**Encoder swapped**: `sentence-transformers/LaBSE` (Apache-2.0, ~471M params, multilingual)
instead of multilingual-E5-small, per your request — no code change needed since `dense.py`
uses plain `AutoModel`/`AutoTokenizer`, which both work the same way for any standard
HF encoder checkpoint.

## What to upload to Kaggle

1. **A code dataset** — a folder (call it e.g. `ber-code-graph`) containing:
   - `src/` = your friend's `block.py`, `cross.py`, `decode.py`, `dense.py`, `features.py`,
     `validate_outputs.py`, **plus the two new files**: `graph.py`, `run_graph.py`
   - `config_graph.json` at the same level as `src/` (not inside it) — same as your
     friend's `config.json`, with `ce_enabled` set to `false` (saves ~35-60 min; flip back
     on once you've confirmed the graph features help) and two new keys, `sib_max_group`
     and `sib_high_cos`.
   - You do **not** need `prep.py`, `dictionaries.json`, or the raw challenge data — you
     already have the prepared parquet, and this notebook validates outputs straight from
     that parquet instead of `validate_outputs.py`'s TSV-based validator.
2. **A prepared-data dataset** (e.g. `ber-prepared-v8`) containing your four parquet files
   — `train_s1.parquet`, `train_q.parquet`, `test_s1.parquet`, `test_q.parquet` — **plus a
   plain-text file named `PREP_VERSION` containing exactly `v8.0`** (no newline needed,
   just that text) so the driver doesn't try to rebuild from raw data.
3. Turn **Internet ON** (this notebook installs `polars`/`rapidfuzz`/`anyascii` from PyPI
   and downloads LaBSE from Hugging Face directly — simpler than your friend's offline
   wheelhouse setup, but if your competition requires Internet **off** for the scored run,
   you'll need to pre-download LaBSE + wheels into an assets dataset the same way your
   friend did with `ber-assets-v6`; happy to help build that once this works end-to-end).
4. Edit the paths in the next code cell, then **Save Version → Save & Run All**.

**Two things worth knowing before a full run:** neither `dense.py` nor `cross.py` shards
work across multiple GPUs (they hardcode a single `cuda` device), so on 2x T4 only one GPU
is used by default — the second sits idle. And per your friend's README, the
blocking/feature-computation stages are CPU-bound, so more/better GPUs don't speed those up
at all. On a weaker single T4 (vs. their apparent RTX 6000), expect the ~4-7 hour estimate
in their README to stretch — likely past a single Kaggle session. The optional dry-run cell
below (a same-idea, parquet-only version of their `tools/make_smoke.py`) lets you confirm
the whole pipeline runs end-to-end on a small slice first, before committing a full session
to the real thing.

In [3]:
from pathlib import Path
import os, subprocess, sys

# ---- Edit these ----
CODE     = Path("/kaggle/input/datasets/arjavjain314/amznmlgraphbasedtry1/graphBased/src")        # folder with run_graph.py, graph.py, block.py, ...
CONFIG   = CODE.parent / "config_graph.json"                # config_graph.json, one level ABOVE src/
PREPARED = Path("/kaggle/input/datasets/arjavjain314/amznmlgraphbasedtry1/graphBased")             # folder with the 4 parquet files + PREP_VERSION
WORK     = Path("/kaggle/temp/ber_graph") if Path("/kaggle/temp").exists() else Path("/kaggle/working/ber_graph_work")
OUT      = Path("/kaggle/working/output")

MODEL_NAME = "sentence-transformers/LaBSE"   # Apache-2.0, multilingual -- swapped in for E5-small

assert (CODE / "run_graph.py").is_file(), f"missing {CODE / 'run_graph.py'}"
assert (CODE / "graph.py").is_file(), f"missing {CODE / 'graph.py'}"
assert CONFIG.is_file(), f"missing {CONFIG}"
assert (PREPARED / "train_s1.parquet").is_file(), f"missing {PREPARED / 'train_s1.parquet'}"
assert (PREPARED / "PREP_VERSION").is_file(), f"missing {PREPARED / 'PREP_VERSION'} (see markdown above)"
assert (PREPARED / "PREP_VERSION").read_text().strip() == "v8.0", "PREP_VERSION must read exactly 'v8.0'"
print("paths OK")

paths OK


## Optional: quick dry run on a slice first

Same idea as your friend's `tools/make_smoke.py`, but applied directly to the parquet you
already have (no raw TSVs needed): keeps a `PERCENT`% slice of S1 entities with all their
labelled queries, the same share of unmatched queries, and a matching slice of the test
files. Run this cell and point `PREPARED` in the cell above at `PREPARED_SMOKE` instead, for
a fast end-to-end check that everything (including the new graph features) actually works,
before spending a full session on the real scale. Skip this cell entirely to go straight to
the full run.

In [4]:
RUN_SMOKE_TEST = True   # flip to True to build a small slice; leave PREPARED above unchanged either way
PERCENT = 2

if RUN_SMOKE_TEST:
    import polars as pl
    PREPARED_SMOKE = Path("/kaggle/working/prepared_smoke")
    PREPARED_SMOKE.mkdir(parents=True, exist_ok=True)
    (PREPARED_SMOKE / "PREP_VERSION").write_text("v8.0")

    keep = lambda c: pl.col(c).hash(seed=5).mod(100) < PERCENT
    s1 = pl.read_parquet(PREPARED / "train_s1.parquet").filter(keep("entity_id"))
    q = pl.read_parquet(PREPARED / "train_q.parquet")
    kept_owner = pl.col("owner").is_in(s1["entity_id"])
    q_small = q.filter(kept_owner | ((pl.col("owner") == "") & keep("entity_id")))
    s1.write_parquet(PREPARED_SMOKE / "train_s1.parquet")
    q_small.write_parquet(PREPARED_SMOKE / "train_q.parquet")

    for name in ("test_s1", "test_q"):
        pl.read_parquet(PREPARED / f"{name}.parquet").filter(keep("entity_id")).write_parquet(PREPARED_SMOKE / f"{name}.parquet")

    print(f"wrote smoke slice to {PREPARED_SMOKE}: "
          f"{s1.height:,} train S1, {q_small.height:,} train queries -- "
          f"point PREPARED at this folder above to use it")

/tmp/ipykernel_58/1202209021.py:14: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  q_small = q.filter(kept_owner | ((pl.col("owner") == "") & keep("entity_id")))


wrote smoke slice to /kaggle/working/prepared_smoke: 44,069 train S1, 205,995 train queries -- point PREPARED at this folder above to use it


In [5]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "polars==1.33.1", "rapidfuzz==3.14.1", "anyascii==0.3.3"], check=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import shutil
import torch
print("GPUs visible:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("free scratch GB:", shutil.disk_usage(WORK.parent if WORK.parent.exists() else "/kaggle").free / 2**30)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 16.0 MB/s eta 0:00:00
GPUs visible: 2 ['Tesla T4', 'Tesla T4']
free scratch GB: 19.469112396240234


## Run

Every stage caches its result under `WORK` (encoder, embeddings, pruned candidates, graph
features, model files, reports) and a rerun skips anything that already exists — same
resume behavior as your friend's pipeline. If a session times out, save `WORK` as a dataset
and copy it back in before rerunning (see the commented line below).

In [ ]:
# Optional resume: copy a previous WORK folder saved as a dataset back in before rerunning.
# shutil.copytree("/kaggle/input/my-ber-graph-work/ber_graph", WORK, dirs_exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-u", str(CODE / "run_graph.py"),
                "--prepared", str(PREPARED),
                "--model", MODEL_NAME,
                "--ce-model", "intfloat/multilingual-e5-small",  # cross-encoder: E5-small (MIT), as the leaders used
                "--work", str(WORK),
                "--out", str(OUT),
                "--config", str(CONFIG)], check=True)

[    0.1 min] train: 2,206,821 S1 (329,947 in E1, 332,417 in E2), 10,320,219 queries
[    0.4 min] encoder: 1,142,576 training pairs (group E); hard negatives from the full catalogue


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1680.17it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[    5.8 min] encoder OOM: batch -> 256
[    5.9 min] encoder OOM: batch -> 128
[    6.0 min] encoder OOM: batch -> 64
[    7.7 min] encoder step 200/17852 loss 0.0236 113s
[    9.6 min] encoder step 400/17852 loss 0.0263 226s
[   11.4 min] encoder step 600/17852 loss 0.0245 338s
[   13.3 min] encoder step 800/17852 loss 0.0261 450s
[   15.2 min] encoder step 1000/17852 loss 0.0111 562s
[   17.0 min] encoder step 1200/17852 loss 0.0062 674s
[   18.9 min] encoder step 1400/17852 loss 0.0652 786s
[   20.8 min] encoder step 1600/17852 loss 0.0407 898s
[   22.6 min] encoder step 1800/17852 loss 0.0460 1011s
[   24.5 min] encoder step 2000/17852 loss 0.0244 1123s
[   26.4 min] encoder step 2200/17852 loss 0.0497 1235s


In [ ]:
import json

print(json.dumps(json.loads((OUT / "run_summary.json").read_text()), indent=2))
# Keep small, useful artefacts in /kaggle/working (models + reports), not the large caches.
for sub in ["reports", "models", "encoder"]:
    if (WORK / sub).exists():
        shutil.copytree(WORK / sub, Path("/kaggle/working/ber_graph_artifacts") / sub, dirs_exist_ok=True)
print("Submit:", OUT / "matching_results.tsv")
print("Keep:", OUT / "candidate_pairs.tsv")

## Validate the outputs

Checks the same rules as the challenge's own validator (full S1 coverage, no duplicate
rows, no duplicate/unknown IDs, matches are a subset of candidates) directly against the
prepared parquet's own `entity_id` sets, since we skipped uploading the raw TSVs.

In [ ]:
import polars as pl

matching = pl.read_csv(OUT / "matching_results.tsv", separator="\t", infer_schema_length=0).fill_null("")
candidates = pl.read_csv(OUT / "candidate_pairs.tsv", separator="\t", infer_schema_length=0).fill_null("")

test_s1_ids = set(pl.read_parquet(PREPARED / "test_s1.parquet", columns=["entity_id"])["entity_id"])
test_q_ids = set(pl.read_parquet(PREPARED / "test_q.parquet", columns=["entity_id"])["entity_id"])

errors = []
if set(matching["source1_entity_id"]) != test_s1_ids:
    errors.append("matching_results.tsv does not cover exactly the test S1 entities")
if matching["source1_entity_id"].is_duplicated().any():
    errors.append("duplicate source1_entity_id rows in matching_results.tsv")

def as_id_lists(col):
    return [[x for x in s.split(",") if x] for s in col.to_list()]

m_ids = as_id_lists(matching["matched_entity_ids"])
c_ids = as_id_lists(candidates["candidate_entity_ids"])
cand_lookup = dict(zip(candidates["source1_entity_id"].to_list(), c_ids))

dup = unknown = not_candidate = 0
for s1_id, ids in zip(matching["source1_entity_id"].to_list(), m_ids):
    if len(ids) != len(set(ids)):
        dup += 1
    if any(i not in test_q_ids for i in ids):
        unknown += 1
    if not set(ids).issubset(set(cand_lookup.get(s1_id, []))):
        not_candidate += 1

print("errors:", errors if errors else "none")
print("rows with a duplicate matched id:", dup)
print("rows with an unknown/out-of-set matched id:", unknown)
print("rows where a match wasn't in the candidate set:", not_candidate)
print("PASS" if not errors and dup == unknown == not_candidate == 0 else "FAIL -- see counts above")